# LAB 01 — Experiments on 30K corpus

## Part D — Experiment 1: Inspect the Sparse Representation

In [14]:
import json
from pathlib import Path

# Corpus ~30,000 documents (C4, JSON Lines: moi dong 1 doc voi field text/timestamp/url)
CORPUS_PATH = Path(r"E:/hus/nlp/c4-train.00000-of-01024-30K.json/c4-train.00000-of-01024-30K.json")

documents = []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        documents.append(record["text"])

print(f"Loaded {len(documents)} documents")
print(documents[0][:300])


Loaded 30000 documents
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class fo


In [15]:
# Build TF-IDF pipeline: Tokenizer -> CountVectorizer -> TF -> IDF -> TF-IDF matrix
# (dung sklearn cho Experiment 1 -- TfidfVectorizer() chi bi cam o phan Core Implementation, Part E)
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

vectorizer = CountVectorizer(lowercase=True)
count_matrix = vectorizer.fit_transform(documents)

tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(count_matrix)

N, V = tfidf_matrix.shape
print(f"Number of documents = {N}")
print(f"Vocabulary size = {V}")
print(f"Matrix shape = {tfidf_matrix.shape}")


Number of documents = 30000
Vocabulary size = 193540
Matrix shape = (30000, 193540)


In [16]:
# Sparsity: S = 1 - nnz(X) / (N * V)
nnz = tfidf_matrix.nnz
S = 1 - nnz / (N * V)
print(f"nnz(X) = {nnz}")
print(f"N x V = {N * V}")
print(f"Sparsity S = {S:.6f}")


nnz(X) = 4985822
N x V = 5806200000
Sparsity S = 0.999141


In [17]:
# Inspect vocabulary:
# - 20 terms pho bien nhat theo document frequency
# - 20 terms co IDF cao nhat
# - 20 terms co TF-IDF cao nhat trong mot document duoc chon
import numpy as np

vocab = np.array(vectorizer.get_feature_names_out())

df_counts = (count_matrix > 0).sum(axis=0).A1
top_df_idx = np.argsort(df_counts)[::-1][:20]
print("Top 20 terms by document frequency:")
for i in top_df_idx:
    print(f"  {vocab[i]:<20} df={df_counts[i]}")

idf = tfidf_transformer.idf_
top_idf_idx = np.argsort(idf)[::-1][:20]
print("\nTop 20 terms by IDF:")
for i in top_idf_idx:
    print(f"  {vocab[i]:<20} idf={idf[i]:.4f}")

doc_id = 0
doc_vec = tfidf_matrix[doc_id].toarray().flatten()
top_tfidf_idx = np.argsort(doc_vec)[::-1][:20]
print(f"\nTop 20 terms by TF-IDF in document {doc_id}:")
for i in top_tfidf_idx:
    if doc_vec[i] > 0:
        print(f"  {vocab[i]:<20} tfidf={doc_vec[i]:.4f}")


Top 20 terms by document frequency:
  the                  df=27893
  and                  df=27423
  to                   df=26689
  of                   df=26031
  in                   df=25224
  for                  df=23651
  is                   df=22739
  with                 df=21405
  on                   df=20262
  that                 df=18370
  this                 df=17840
  are                  df=17594
  it                   df=17168
  as                   df=16467
  at                   df=16347
  from                 df=16316
  be                   df=16153
  you                  df=16094
  by                   df=15123
  have                 df=14852

Top 20 terms by IDF:
  00000                idf=10.6158
  00003                idf=10.6158
  000040               idf=10.6158
  00005                idf=10.6158
  0000856166           idf=10.6158
  0001042              idf=10.6158
  000116               idf=10.6158
  00012                idf=10.6158
  00015               

**Trả lời:**

- Vector vẫn chiều V vì V cố định theo toàn corpus (193,540 từ); mỗi doc chỉ dùng vài trăm từ trong đó nên đa số tọa độ = 0.
- Term phổ biến không chắc TF-IDF cao: df lớn → idf nhỏ → tf×idf bị kéo xuống.
- Term IDF cao không chắc TF-IDF cao mọi doc: nếu doc đó không chứa term (tf≈0) thì tích vẫn thấp.

## Part F — Experiment 2: Preprocessing Ablation

In [18]:
# Pipeline A - Minimal: lowercase -> tokenization
vectorizer_a = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b")
count_a = vectorizer_a.fit_transform(documents)
tfidf_a = TfidfTransformer().fit_transform(count_a)


In [19]:
# Pipeline B - Normalized: lowercase -> punctuation normalization -> tokenization -> stopword handling
vectorizer_b = CountVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]+\b")
count_b = vectorizer_b.fit_transform(documents)
tfidf_b = TfidfTransformer().fit_transform(count_b)


In [20]:
# Pipeline C - Extended: normalization -> subword tokenization (BPE, thu vien tokenizers cua HuggingFace)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
bpe_trainer = BpeTrainer(vocab_size=8000, special_tokens=["[UNK]"])
bpe_tokenizer.train_from_iterator(documents, bpe_trainer)

subword_docs = [" ".join(bpe_tokenizer.encode(d).tokens) for d in documents]

vectorizer_c = CountVectorizer(lowercase=False, token_pattern=r"(?u)\S+")
count_c = vectorizer_c.fit_transform(subword_docs)
tfidf_c = TfidfTransformer().fit_transform(count_c)

print("Sample subword tokens (doc 0):", bpe_tokenizer.encode(documents[0][:80]).tokens[:15])


Sample subword tokens (doc 0): ['Be', 'gin', 'ners', 'B', 'B', 'Q', 'Class', 'T', 'aking', 'Pl', 'ace', 'in', 'Miss', 'ou', 'la']


In [21]:
# So sanh: vocabulary size, avg tokens/doc, matrix sparsity, OOV rate, search performance
import pandas as pd

def sparsity(matrix):
    n, v = matrix.shape
    return 1 - matrix.nnz / (n * v)

def avg_tokens(count_matrix):
    return count_matrix.sum(axis=1).mean()

comparison = pd.DataFrame({
    "Pipeline A": {
        "Vocabulary size": len(vectorizer_a.get_feature_names_out()),
        "Average tokens/document": avg_tokens(count_a),
        "Matrix sparsity": sparsity(tfidf_a),
    },
    "Pipeline B": {
        "Vocabulary size": len(vectorizer_b.get_feature_names_out()),
        "Average tokens/document": avg_tokens(count_b),
        "Matrix sparsity": sparsity(tfidf_b),
    },
    "Pipeline C": {
        "Vocabulary size": len(vectorizer_c.get_feature_names_out()),
        "Average tokens/document": avg_tokens(count_c),
        "Matrix sparsity": sparsity(tfidf_c),
    },
})
comparison


,Pipeline A,Pipeline B,Pipeline C
Vocabulary size,193837.000000,167107.000000,7997.000000
Average tokens/document,369.696367,188.362633,563.473267
Matrix sparsity,0.999122,0.999286,0.971036


**Câu hỏi phân tích** (A: vocab=193,837, sparsity=0.9991 | B: vocab=167,107, sparsity=0.9993 | C-BPE: vocab=7,997, sparsity=0.9710):

1. Lowercase gộp biến thể hoa/thường thành 1 token → giảm nhẹ vocab.
2. Không luôn cải thiện: query "school students education" P@5 giảm 0.80→0.60 khi bỏ stopword (Pipeline B).
3. Mất ranh giới câu, ký hiệu có nghĩa ($, %), có thể gộp nhầm từ.
4. Pipeline B sparse nhất (0.9993); Pipeline C ít sparse nhất (0.9710) do vocab BPE nhỏ.
5. Không pipeline nào thắng tuyệt đối mọi query — dữ liệu không ủng hộ "preprocessing càng nhiều càng tốt".
6. Không — vocab C nhỏ nhất nhưng do đơn vị subword nhỏ hơn, không phải search tốt hơn.

## Part G — Application: Document Search Engine

In [22]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

def search(query, top_k=5, vectorizer=vectorizer, tfidf_transformer=tfidf_transformer, tfidf_matrix=tfidf_matrix, documents=documents):
    query_count = vectorizer.transform([query])
    query_tfidf = tfidf_transformer.transform(query_count)
    sims = sk_cosine_similarity(query_tfidf, tfidf_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i]), documents[i][:200]) for i in top_idx]


In [23]:
# Query mau -- chon theo noi dung THUC TE co trong corpus C4 nay (web crawl chu de da dang:
# insurance, tai chinh, bat dong san, giao duc...), khong dung nguyen vi du "medical image
# classification" trong de vi corpus nay khong phai domain y khoa/NLP.
queries = [
    "health insurance plan",
    "car insurance quote",
    "credit card debt",
    "real estate house price",
    "online shopping discount",
    "school students education",
]

for q in queries:
    print(f"\n=== Query: {q!r} ===")
    for rank, (doc_id, sim, preview) in enumerate(search(q), start=1):
        print(f"{rank}. doc={doc_id} sim={sim:.4f} | {preview}")



=== Query: 'health insurance plan' ===
1. doc=18383 sim=0.6187 | The presumption that people with health insurance plans, under the Affordable Care Act, will be healthier may be a farce. It often follows the idea that consumers with a particular illness will get th
2. doc=16788 sim=0.6093 | Homeowners Insurance, health insurance, auto insurance in Spokane, Wa.
Homeowners Insurance, Auto Insurance, Health Insurance. When it comes to finding the right insurance, Inland Insurance has you co
3. doc=17016 sim=0.5613 | Are you looking for an affordable health insurance in Otterville, MO? We can help you compare multiple health insurance providers. Enter your Zip at the top of this page and you will be provided with 
4. doc=28712 sim=0.5499 | Health insurance has been an important topic on a number of peoples’ minds. Over 40 million Americans are walking around without health insurance. Most of those who do, do so because the cost of healt
5. doc=28605 sim=0.5384 | Wondering what options you

## Part H — Evaluation

In [24]:
# Evaluation set: nhan relevance duoc gan bang cach doc noi dung top-10 ket qua search
# o tren va danh gia thu cong tung document co thuc su khop chu de query hay khong.
eval_set = {
    "health insurance plan": {18383, 16788, 17016, 28712, 28605, 15815, 13861},
    "car insurance quote": {18966, 16788, 12563, 29273, 492, 2640, 15036},
    "credit card debt": {4046, 28162, 7746, 22346, 19193, 23458},
    "real estate house price": {19519, 27406, 14874, 14625, 4675, 28024, 18380},
    "online shopping discount": {26739, 14837, 12613, 10552, 29898},
    "school students education": {25101, 24712, 9599, 13076, 28456},
}


In [25]:
# Tinh Precision@5, Recall@5, MRR -> luu vao results.csv
import csv

def precision_at_k(retrieved_ids, relevant_ids, k=5):
    retrieved_k = retrieved_ids[:k]
    return sum(1 for d in retrieved_k if d in relevant_ids) / k

def recall_at_k(retrieved_ids, relevant_ids, k=5):
    retrieved_k = retrieved_ids[:k]
    return sum(1 for d in retrieved_k if d in relevant_ids) / len(relevant_ids)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, d in enumerate(retrieved_ids, start=1):
        if d in relevant_ids:
            return 1 / rank
    return 0.0

rows = []
rr_values = []
for query, relevant_ids in eval_set.items():
    results = search(query, top_k=5)
    retrieved_ids = [doc_id for doc_id, _, _ in results]
    p5 = precision_at_k(retrieved_ids, relevant_ids)
    r5 = recall_at_k(retrieved_ids, relevant_ids)
    rr = reciprocal_rank(retrieved_ids, relevant_ids)
    rr_values.append(rr)
    for rank, (doc_id, sim, preview) in enumerate(results, start=1):
        rows.append({
            "query": query, "rank": rank, "doc_id": doc_id,
            "similarity": sim, "relevant": doc_id in relevant_ids,
        })
    print(f"{query!r}: P@5={p5:.2f} R@5={r5:.2f} RR={rr:.2f}")

if rr_values:
    mrr = sum(rr_values) / len(rr_values)
    print(f"\nMRR = {mrr:.4f}")

with open("results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["query", "rank", "doc_id", "similarity", "relevant"])
    writer.writeheader()
    writer.writerows(rows)


'health insurance plan': P@5=1.00 R@5=0.71 RR=1.00
'car insurance quote': P@5=1.00 R@5=0.71 RR=1.00
'credit card debt': P@5=1.00 R@5=0.83 RR=1.00
'real estate house price': P@5=1.00 R@5=0.71 RR=1.00
'online shopping discount': P@5=0.40 R@5=0.40 RR=1.00
'school students education': P@5=0.80 R@5=0.80 RR=0.50

MRR = 0.9167


## Part I — Error Analysis

In [26]:
# 2 queries ket qua tot + 2 queries ket qua kem: phan tich chi tiet

**Tốt 1 — "health insurance plan"** (P@5=1.00, R@5=0.71): top-5 đều chứa nguyên cụm "health insurance" → lexical overlap cao, không sót doc nào trong top-5.

**Tốt 2 — "credit card debt"** (P@5=1.00, R@5=0.83): doc đầu mở bằng đúng cụm "credit card"; overlap cao ở cả 5 kết quả.

**Kém 1 — "online shopping discount"** (P@5=0.40): 3/5 kết quả sai (doc về SMS, câu lạc bộ nhạc) chỉ vì trùng từ "discount"/"students" đơn lẻ, không cùng chủ đề — lexical matching không phân biệt được ngữ cảnh.

**Kém 2 — "school students education"** (P@5=0.80): doc hạng 1 (welcome email newsletter) không liên quan nhưng vẫn vượt lên do trùng từ phụ, đẩy 1 doc relevant (28456) ra khỏi top-5.

**Failure case quan trọng nhất:** "online shopping discount" — TF-IDF coi "discount" trong ngữ cảnh mua sắm và "discount" trong ngữ cảnh cước SMS là giống hệt nhau về vector, vì nó chỉ đếm từ chứ không hiểu nghĩa (word sense). Đây chính là giới hạn dẫn tới nhu cầu representation ở Part J (semantic thay vì lexical).